In [1]:
import xarray as xr
import dask.array as da
import numpy as np

def create_sample_dataset(chunks: tuple) -> xr.Dataset:
    # Define dimensions and coordinates
    time = np.arange(218)
    X = np.linspace(-2.19e4, 1.059e4, 1084)
    Y = np.linspace(7.65e4, 1.418e5, 2178)

    # Create Dask arrays for the data variables
    SR_B4 = da.random.random((218, 1084, 2178), chunks=chunks)
    SR_B5 = da.random.random((218, 1084, 2178), chunks=chunks)

    # Create xarray Dataset
    dataset = xr.Dataset({
        'SR_B4': (['time', 'X', 'Y'], SR_B4),
        'SR_B5': (['time', 'X', 'Y'], SR_B5),
    }, coords={'time': time, 'X': X, 'Y': Y})

    #print(dataset)
    #total_size_mb = sum(var.nbytes for var in dataset.data_vars.values()) / (1024 * 1024 * 1024)
    #print("Approximate size of the dataset:", total_size_mb, "GBs")
    return dataset

In [20]:
import time
from dask.distributed import Client

def benchmark(n_workers, threads_per_worker, memory_limit: str, chunks = (128, 128, 128)):
    client = Client(n_workers=n_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
    try:
        dataset = create_sample_dataset(chunks)
        
        start_time = time.time()
        ndvi = (dataset['SR_B5'] - dataset['SR_B4']) / (dataset['SR_B5'] + dataset['SR_B4'])
        dataset['NDVI'] = ndvi
        result = dataset.compute()
        #%%time
        end_time = time.time()
        
        client.close()
        
        elapsed_time = end_time - start_time
        print(f"Config: {n_workers} workers, {threads_per_worker} threads each, Time: {elapsed_time:.2f} seconds")
        return elapsed_time
    except Exception as e:
        client.close()

#time1 = benchmark(n_workers=8, threads_per_worker=1)
#time2 = benchmark(n_workers=4, threads_per_worker=2)

#print(f"Single-threaded workers: {time1:.2f} seconds")
#print(f"Multi-threaded workers: {time2:.2f} seconds")


In [ ]:
time1 = benchmark(n_workers=4, threads_per_worker=1, memory_limit='16GB')

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\node.py:183: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 54753 instead
  warnings.warn(
2024-05-22 12:04:27,901 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54782 (pid=44220) exceeded 95% memory budget. Restarting...


Memory budget exceeded. Closing client.


2024-05-22 12:04:30,081 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54791 (pid=33000) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:30,083 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54785 (pid=37416) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:30,085 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54786 (pid=56416) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:30,258 - distributed.scheduler - ERROR - Couldn't gather keys {"('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 0, 4, 4)": ['tcp://127.0.0.1:54785'], "('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 4, 6)": ['tcp://127.0.0.1:54786'], "('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 1, 7, 10)": ['tcp://127.0.0.1:54786'], "('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 6, 13)": ['tcp://127.0.0.1:54785'], "('truediv-bb5d8e457dd503856adee6a4b88f6516', 0, 5, 6)": ['tcp://127.0.0.1:54782'], "('random_sample-21a7cd398a682a25

Memory budget exceeded. Closing client.
Memory budget exceeded. Closing client.
Memory budget exceeded. Closing client.


2024-05-22 12:04:30,283 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54786'], ('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 4, 6)
NoneType: None
2024-05-22 12:04:30,290 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54786'], ('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 1, 7, 10)
NoneType: None
2024-05-22 12:04:30,291 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54785'], ('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 6, 13)
NoneType: None
2024-05-22 12:04:30,291 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54782'], ('truediv-bb5d8e457dd503856adee6a4b88f6516', 0, 5, 6)
NoneType: None
2024-05-22 12:04:30,292 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54791'], ('random_sample-21a7cd398a682a25cd42e75

...

2024-05-22 12:04:48,536 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54839 (pid=49880) exceeded 95% memory budget. Restarting...


Memory budget exceeded. Closing client.


2024-05-22 12:04:50,045 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54852 (pid=48924) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:50,046 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54857 (pid=55352) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:50,047 - distributed.worker_memory - WARNING - Worker tcp://127.0.0.1:54851 (pid=4808) exceeded 95% memory budget. Restarting...
2024-05-22 12:04:50,219 - distributed.scheduler - ERROR - Couldn't gather keys {"('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 0, 4, 4)": ['tcp://127.0.0.1:54851'], "('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 4, 6)": ['tcp://127.0.0.1:54852'], "('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 1, 7, 10)": ['tcp://127.0.0.1:54857', 'tcp://127.0.0.1:54852'], "('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 6, 13)": ['tcp://127.0.0.1:54857'], "('truediv-bb5d8e457dd503856adee6a4b88f6516', 0, 5, 6)": ['tcp://127.0.0.1:54857'], "('random

Memory budget exceeded. Closing client.
Memory budget exceeded. Closing client.
Memory budget exceeded. Closing client.


2024-05-22 12:04:50,294 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54851'], ('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 0, 4, 4)
NoneType: None
2024-05-22 12:04:50,296 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54852'], ('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 4, 6)
NoneType: None
2024-05-22 12:04:50,297 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54857', 'tcp://127.0.0.1:54852'], ('random_sample-8ccb75faa798c6596a6f6608fdb639b6', 1, 7, 10)
NoneType: None
2024-05-22 12:04:50,298 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54857'], ('random_sample-21a7cd398a682a25cd42e75dbc91a62e', 0, 6, 13)
NoneType: None
2024-05-22 12:04:50,298 - distributed.scheduler - ERROR - Shut down workers that don't have promised key: ['tcp://127.0.0.1:54857'], ('truedi

In [27]:
import time
import rioxarray
from dask.distributed import LocalCluster

chunks = (256, 256, 256)
n_workers = 4
threads_per_worker = 1
memory_limit = "8GB"

dataset = create_sample_dataset(chunks)

cluster = LocalCluster(n_workers = n_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)          # Fully-featured local Dask cluster
client = cluster.get_client()

start_time = time.time()
ndvi = (dataset['SR_B5'] - dataset['SR_B4']) / (dataset['SR_B5'] + dataset['SR_B4'])
dataset['NDVI'] = ndvi
dataset = dataset.rename({'X': 'x', 'Y': 'y'})
dataset = dataset.transpose('time', 'y', 'x')
dataset.isel(time=0).rio.to_raster('output.tif')
#result = client.persist(dataset)
#%%time

end_time = time.time()  
elapsed_time = end_time - start_time
print(f"Config: {n_workers} workers, {threads_per_worker} threads each, Time: {elapsed_time:.2f} seconds")

Config: 4 workers, 1 threads each, Time: 7.54 seconds


2024-05-23 18:10:19,299 - distributed.utils_perf - WARNING - full garbage collections took 12% CPU time recently (threshold: 10%)
2024-05-23 18:12:48,587 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [25]:
client.close()

In [28]:
cluster.close()

In [12]:
print(result)

<xarray.Dataset>
Dimensions:  (time: 218, X: 1084, Y: 2178)
Coordinates:
  * time     (time) int32 0 1 2 3 4 5 6 7 8 ... 210 211 212 213 214 215 216 217
  * X        (X) float64 -2.19e+04 -2.187e+04 -2.184e+04 ... 1.056e+04 1.059e+04
  * Y        (Y) float64 7.65e+04 7.653e+04 7.656e+04 ... 1.418e+05 1.418e+05
Data variables:
    SR_B4    (time, X, Y) float64 dask.array<chunksize=(218, 256, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float64 dask.array<chunksize=(218, 256, 256), meta=np.ndarray>
    NDVI     (time, X, Y) float64 dask.array<chunksize=(218, 256, 256), meta=np.ndarray>


In [ ]:
import sys
import os

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from initialize_dask import DaskHandler

handler = DaskHandler()
handler.create_local_cluster(n_workers=4, threads_per_worker=1, memory_limit="1GB")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da

# Create some time data
times = pd.date_range('2000-01-01', periods=24, freq='M')

# Create some spatial data (latitude and longitude)
latitudes = np.linspace(-90, 90, 256)
longitudes = np.linspace(-180, 180, 128)

# Create random data
data = da.random.random(size=(24, 256, 128), chunks=(4, 64, 32))

# Create the xarray Dataset
ds = xr.Dataset(
    {
        'variable1': (['time', 'lat', 'lon'], data),
        'variable2': (['time', 'lat', 'lon'], data * 2)
    },
    coords={
        'time': times,
        'lat': latitudes,
        'lon': longitudes
    }
)

# Ensure the Dataset is Dask-backed
ds = ds.chunk({'time': 4, 'lat': 64, 'lon': 32})

print(ds)
